# MCU 毛利率提取
从 GCS 年报 PDF 中提取各公司 MCU 分部毛利率，写回 `mcu_known_data.json` 并推送 git。

**使用前**：
1. 点击菜单 Runtime → Run all，或逐格运行
2. Cell 2 会弹出 Google 账号授权（需有 `st-china-ai-force` 项目权限）
3. Cell 7 需要 GitHub PAT（Settings → Developer settings → Personal access tokens）

In [ ]:
# Cell 1 — 安装依赖
!pip install -q google-cloud-storage google-genai vertexai

In [ ]:
# Cell 2 — Google 认证（使用 Vertex AI，无需 API Key）
from google.colab import auth
auth.authenticate_user()

import vertexai
PROJECT = 'st-china-ai-force'
LOCATION = 'us-central1'
vertexai.init(project=PROJECT, location=LOCATION)
print('✓ 认证完成')

In [ ]:
# Cell 3 — 克隆仓库，读取 mcu_known_data.json
import subprocess, json, os

REPO_URL = 'https://github.com/Yude-Jiang/mcu-regional-competitor-dashboard.git'
BRANCH   = 'claude/loving-wozniak-GQeKA'
REPO_DIR = '/content/mcu-dash'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

with open(f'{REPO_DIR}/mcu_known_data.json') as f:
    known = json.load(f)
with open(f'{REPO_DIR}/companies_meta.json') as f:
    meta = json.load(f)

print(f'✓ 仓库同步完成（分支: {BRANCH}）')

In [ ]:
# Cell 4 — 构建待提取列表

BUCKET = 'st-finance-reports'

# MCU 分部关键词（用于告诉 Gemini 找哪一行）
SEGMENT = {
    '603986': '微控制器（MCU及模拟产品）',
    '300327': '工业控制芯片',
    '688380': None,   # 单一产品，用全公司毛利率
    '300077': 'MCU芯片',
    '688279': None,   # total_proxy，用全公司毛利率
    '002180': '芯片',
    '688385': '智能电表芯片',
    '688766': 'MCU（微控制器）',
    '688595': 'MCU芯片',
    '688391': None,   # total_proxy，用全公司毛利率
    '688018': '芯片',
}

# 只处理有营收数据但缺毛利率的条目
worklist = []
for sym, yrs in known.items():
    for yr, yd in yrs.items():
        if not isinstance(yd, dict): continue
        has_rev  = yd.get('mcu_revenue_yuan') is not None
        has_gm   = yd.get('mcu_gross_margin') is not None
        if has_rev and not has_gm:
            worklist.append((sym, int(yr)))

worklist.sort()
print(f'待提取条目: {len(worklist)} 个')
for s, y in worklist:
    name = meta.get(s, {}).get('name_cn', s)
    seg  = SEGMENT.get(s) or '全公司综合'
    print(f'  {s} {name} {y}  分部=「{seg}」')

In [ ]:
# Cell 5 — 提取函数（Gemini 直读 PDF bytes，Vertex AI 认证）

import io, re
from google.cloud import storage
from vertexai.generative_models import GenerativeModel, Part

gcs = storage.Client(project=PROJECT)
model = GenerativeModel('gemini-2.0-flash-001')

PROMPT = """你是一位专业的财务分析师。请从这份年报PDF中完成以下任务：

在「主营业务分产品情况」表（或「主营业务分行业情况」表）中，找到「{segment}」行。
表格列顺序通常为：产品/行业名称 | 营业收入 | 营业成本 | 毛利率(%) | ...

{extra}

严格按以下 JSON 格式返回，不要输出其他内容：
{{
  "gross_margin_pct": <数字如36.07，不要加%符号，找不到则null>,
  "revenue_yuan": <营业收入元，整数，如无则null>,
  "source_text": "<原文关键片段，限80字>",
  "page": <页码整数或null>
}}"""

EXTRA_TOTAL = '注意：该公司无单独MCU产品行，请返回集成电路/芯片业务整体的毛利率。'
EXTRA_SEG   = '注意：请只返回「{segment}」这一行的毛利率，不要返回全公司综合毛利率。'

def find_blob(sym, year):
    bucket = gcs.bucket(BUCKET)
    for blob in bucket.list_blobs(prefix='reports/'):
        n = blob.name
        if sym in n and str(year) in n and n.endswith('.pdf'):
            return blob
    return None

def extract_one(sym, year):
    seg  = SEGMENT.get(sym)
    blob = find_blob(sym, year)
    if blob is None:
        return None, f'PDF not found ({sym} {year})'

    buf = io.BytesIO()
    blob.download_to_file(buf)
    buf.seek(0)

    extra = EXTRA_TOTAL if seg is None else EXTRA_SEG.format(segment=seg)
    prompt = PROMPT.format(segment=seg or '整体业务', extra=extra)

    resp = model.generate_content([
        Part.from_data(data=buf.read(), mime_type='application/pdf'),
        prompt,
    ])

    text = resp.text.strip()
    # Strip markdown code fences if present
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)

    try:
        result = json.loads(text)
        gm  = result.get('gross_margin_pct')
        src = result.get('source_text', '')
        pg  = result.get('page')
        return gm, f'第{pg}页 {src}' if pg else src
    except Exception as e:
        return None, f'JSON parse error: {text[:120]}'

print('✓ 函数定义完成')

In [ ]:
# Cell 6 — 批量提取（逐条打印结果，方便人工核对）

results = {}   # (sym, yr) -> {'gm': float|None, 'src': str}

for sym, yr in worklist:
    name = meta.get(sym, {}).get('name_cn', sym)
    seg  = SEGMENT.get(sym) or '全公司'
    print(f'\n▶ {sym} {name} {yr}  [{seg}]')
    try:
        gm, src = extract_one(sym, yr)
        results[(sym, yr)] = {'gm': gm, 'src': src}
        label = f'{gm:.2f}%' if gm is not None else 'null'
        print(f'  毛利率: {label}')
        print(f'  来源:   {str(src)[:100]}')
    except Exception as e:
        results[(sym, yr)] = {'gm': None, 'src': f'ERROR: {e}'}
        print(f'  ERROR: {e}')

# 汇总
print('\n' + '='*70)
print('汇总')
print(f'{"symbol":<8} {"year"} {"gross_margin":>12}  source')
print('-'*70)
for (sym, yr), v in sorted(results.items()):
    gm  = v['gm']
    src = str(v['src'])[:55]
    lbl = f'{gm:.2f}%' if gm is not None else 'null'
    name = meta.get(sym, {}).get('name_cn', sym)
    print(f'{sym} {name:<6} {yr}  {lbl:>8}   {src}')

In [ ]:
# Cell 7 — 写回 mcu_known_data.json 并推送 git
# 运行前请确认 Cell 6 的结果无误

# ── 写入 JSON ──────────────────────────────────────────────
with open(f'{REPO_DIR}/mcu_known_data.json') as f:
    d = json.load(f)

written = []
for (sym, yr), v in results.items():
    gm = v['gm']
    if gm is None:
        continue
    yr_str = str(yr)
    if sym not in d or yr_str not in d[sym]:
        continue
    gm_decimal = round(gm / 100 if gm > 1 else gm, 4)
    d[sym][yr_str]['mcu_gross_margin'] = gm_decimal
    # 补充来源说明
    old_src = d[sym][yr_str].get('source', '')
    if '毛利率' not in old_src:
        d[sym][yr_str]['source'] = old_src + f'；毛利率{gm:.2f}%来自分产品表'
    written.append(f'{sym} {yr} → {gm_decimal}')

with open(f'{REPO_DIR}/mcu_known_data.json', 'w') as f:
    json.dump(d, f, ensure_ascii=False, indent=2)

print(f'写入 {len(written)} 条:')
for w in written: print(f'  {w}')

# ── Git commit + push ──────────────────────────────────────
PAT = input('\nGitHub PAT（用于 push，留空则跳过）: ').strip()

if PAT:
    cmds = [
        ['git', '-C', REPO_DIR, 'config', 'user.email', 'colab-extract@noreply'],
        ['git', '-C', REPO_DIR, 'config', 'user.name',  'Colab Extractor'],
        ['git', '-C', REPO_DIR, 'add', 'mcu_known_data.json'],
        ['git', '-C', REPO_DIR, 'commit', '-m',
         f'feat(data): extract MCU gross margins via Colab ({len(written)} entries)'],
    ]
    for cmd in cmds:
        subprocess.run(cmd, check=True)

    push_url = f'https://{PAT}@github.com/Yude-Jiang/mcu-regional-competitor-dashboard.git'
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'push', push_url, BRANCH],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print('✓ Push 成功')
    else:
        print('Push 失败:', r.stderr)
else:
    print('跳过 push — 请手动从 Cloud Shell 拉取并推送')